In [64]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd() / "estonia-exam-gap"
RAW = ROOT / "data" / "raw"                # the downloaded Haridussilm / Statistikaamet files (never modified)
EXT = ROOT / "data" / "external"           # cached web look-ups (coordinates, road distances)
PROCESSED = ROOT / "data" / "processed"    # tables this notebook writes


In [65]:
!git clone 'https://github.com/valpar/estonia-exam-gap.git'

fatal: destination path 'estonia-exam-gap' already exists and is not an empty directory.


# X1 : Qualified Teacher Ratio

Ideal Format (Columns)

school_name,
exam_year,
estonian_proficiency_ratio,
teaching_qualification_ratio,

In [66]:
df_raw_qualified_teacher = pd.read_excel(ROOT/RAW/'haridussilm_qualified_teachers_2021-2025_by_school.xlsx', header=2)
df_raw_qualified_teacher.head()

,Indikaator,Õppeasutus/KOV/Eesti,'Date'[Year],Väärtused
0,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2021,100
1,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2022,100
2,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2023,100
3,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2024,"95,5"
4,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2025,100


In [67]:
# Examine the columns
df_raw_qualified_teacher.columns

Index(['Indikaator', 'Õppeasutus/KOV/Eesti', ''Date'[Year]', 'Väärtused'], dtype='object')

In [68]:
# Translation of estonian column names to english
df_raw_qualified_teacher = df_raw_qualified_teacher.rename(columns={
    "Õppeasutus/KOV/Eesti": "school",
    "'Date'[Year]": "year",
    "Indikaator": "indicator",
    "Väärtused": "percentage"
})
df_raw_qualified_teacher.head()

,indicator,school,year,percentage
0,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2021,100
1,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2022,100
2,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2023,100
3,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2024,"95,5"
4,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2025,100


In [69]:
# Convert percentatge to decimal values
df_raw_qualified_teacher['percentage'] = df_raw_qualified_teacher['percentage'].astype(str).str.replace(',', '.').astype(float)

# Pivot the table to mathcing ideal format definted above
df_raw_qualified_teacher_wide = pd.pivot_table(
    df_raw_qualified_teacher,
    index=['school', 'year'],
    columns='indicator',
    values='percentage'
).reset_index()

# Rename the columns
df_raw_qualified_teacher_wide.columns = [
    'school_name',
    'exam_year',
    'estonian_proficiency_ratio',
    'teaching_qualification_ratio'
]

df_raw_qualified_teacher_wide.head()

,school_name,exam_year,estonian_proficiency_ratio,teaching_qualification_ratio
0,''Eduson'' kool,2021,80.0,100.0
1,''Eduson'' kool,2022,68.8,100.0
2,''Eduson'' kool,2023,60.0,100.0
3,''Eduson'' kool,2024,85.7,100.0
4,''Eduson'' kool,2025,100.0,100.0


# X2 : State of the School (State or Municipal)

Ideal Format (Columns)

school_name, ownership, academic_year

In [70]:
# Load schools ran by state
df_raw_school_state = pd.read_excel(ROOT/RAW/'/content/estonia-exam-gap/data/raw/haridussilm_students_2021-2025_state.xlsx', header=2)
df_raw_school_state.info()
df_raw_school_state.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163 entries, 0 to 162
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   KOV/Õppeasutus   163 non-null    object
 1   Õppeasutus       163 non-null    object
 2   Õppurite arv ap  163 non-null    int64 
 3   Õppeaasta        163 non-null    object
dtypes: int64(1), object(3)
memory usage: 5.2+ KB


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,KOV/Õppeasutus,Õppeasutus,Õppurite arv ap,Õppeaasta
0,Haapsalu linn,Läänemaa Ühisgümnaasium,258,2020/21
1,Haapsalu linn,Läänemaa Ühisgümnaasium,272,2021/22
2,Haapsalu linn,Läänemaa Ühisgümnaasium,283,2022/23
3,Haapsalu linn,Läänemaa Ühisgümnaasium,260,2023/24
4,Haapsalu linn,Läänemaa Ühisgümnaasium,268,2024/25


In [71]:
# Load schools ran by municipal
df_raw_school_municipal = pd.read_excel(ROOT/RAW/'/content/estonia-exam-gap/data/raw/haridussilm_students_2021-2025_municipal.xlsx', header=2)
df_raw_school_municipal.info()
df_raw_school_municipal.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 631 entries, 0 to 630
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   KOV/Õppeasutus   631 non-null    object
 1   Õppeasutus       631 non-null    object
 2   Õppurite arv ap  631 non-null    int64 
 3   Õppeaasta        631 non-null    object
dtypes: int64(1), object(3)
memory usage: 19.8+ KB


,KOV/Õppeasutus,Õppeasutus,Õppurite arv ap,Õppeaasta
0,Alutaguse vald,Iisaku Põhikool,48,2020/21
1,Alutaguse vald,Iisaku Põhikool,50,2021/22
2,Alutaguse vald,Iisaku Põhikool,31,2022/23
3,Alutaguse vald,Iisaku Põhikool,20,2023/24
4,Alutaguse vald,Iisaku Põhikool,7,2024/25


In [72]:
# Add ownership label for two different type of school
df_raw_school_state.insert(1, 'ownership', value='state')
df_raw_school_municipal.insert(1, 'ownership', value='municipal')

# Merge school names (Õppeasutus) from both table and give state/municipal labels
df_raw_school_combined = pd.concat([df_raw_school_municipal, df_raw_school_state], ignore_index=True)
df_raw_school_combined.info()
df_raw_school_combined.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 794 entries, 0 to 793
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   KOV/Õppeasutus   794 non-null    object
 1   ownership        794 non-null    object
 2   Õppeasutus       794 non-null    object
 3   Õppurite arv ap  794 non-null    int64 
 4   Õppeaasta        794 non-null    object
dtypes: int64(1), object(4)
memory usage: 31.1+ KB


,KOV/Õppeasutus,ownership,Õppeasutus,Õppurite arv ap,Õppeaasta
0,Alutaguse vald,municipal,Iisaku Põhikool,48,2020/21
1,Alutaguse vald,municipal,Iisaku Põhikool,50,2021/22
2,Alutaguse vald,municipal,Iisaku Põhikool,31,2022/23
3,Alutaguse vald,municipal,Iisaku Põhikool,20,2023/24
4,Alutaguse vald,municipal,Iisaku Põhikool,7,2024/25


In [73]:
# Rename table column propery
df_raw_school_combined = df_raw_school_combined[['Õppeasutus', 'ownership','Õppeaasta']]
df_raw_school_combined.columns = [
    'school_name',
    'ownership',
    'academic_year'
]
df_raw_school_combined.head()

,school_name,ownership,academic_year
0,Iisaku Põhikool,municipal,2020/21
1,Iisaku Põhikool,municipal,2021/22
2,Iisaku Põhikool,municipal,2022/23
3,Iisaku Põhikool,municipal,2023/24
4,Iisaku Põhikool,municipal,2024/25




# X3 : Distance to Hubs

Ideal Format (Columns)

school_name, longitude, latitude, distance_nearest_hub



In [74]:
# Install tools for distance calculation
!pip install geopandas shapley pyrosm

# Download latest osm data for estonia
!wget https://download.geofabrik.de/europe/estonia-latest.osm.pbf

--2026-11-02 14:25:53--  https://download.geofabrik.de/europe/estonia-latest.osm.pbf
Resolving download.geofabrik.de (download.geofabrik.de)... 95.217.63.98, 95.216.245.185, 95.216.245.233, ...
Connecting to download.geofabrik.de (download.geofabrik.de)|95.217.63.98|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: https://download.geofabrik.de/europe/estonia-261101.osm.pbf [following]
--2026-11-02 14:26:09--  https://download.geofabrik.de/europe/estonia-261101.osm.pbf
Reusing existing connection to download.geofabrik.de:443.
HTTP request sent, awaiting response... 200 OK
Length: 124211622 (118M) [application/octet-stream]
Saving to: ‘estonia-latest.osm.pbf.1’

estonia-latest.osm. 100%[===================>] 118.46M  1.02MB/s    in 4m 25s  

2026-11-02 14:30:35 (457 KB/s) - ‘estonia-latest.osm.pbf.1’ saved [124211622/124211622]



In [75]:
# Extract a school list
df_raw_school_list = df_raw_school_combined[['school_name']].drop_duplicates()
df_raw_school_list.info()
df_raw_school_list.head()

<class 'pandas.core.frame.DataFrame'>
Index: 150 entries, 0 to 788
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   school_name  150 non-null    object
dtypes: object(1)
memory usage: 2.3+ KB


,school_name
0,Iisaku Põhikool
5,Kehra Gümnaasium
11,Antsla Gümnaasium
17,Elva Gümnaasium
23,Rõngu Keskkool


In [76]:
# Fetch geo location of all school from estonia latest Open Stree Map data
from pyrosm import OSM

osm = OSM("/content/estonia-latest.osm.pbf")
schools = osm.get_pois(
    custom_filter={
        "amenity": ["school"]
    }
)
schools = schools[schools["name"].notna()]

# Geo data transformation (From help of AI)
schools_projected = schools.to_crs(epsg=3301)
centroids = schools_projected.geometry.centroid
centroids_wgs84 = centroids.to_crs(epsg=4326)

schools["longitude"] = centroids_wgs84.x
schools["latitude"] = centroids_wgs84.y
df_raw_schools_coordination = schools[["name", "latitude", "longitude"]].copy()
df_raw_schools_coordination.columns = ["school_name", "latitude", "longitude"]
df_raw_schools_coordination.info()
df_raw_schools_coordination.head()

<class 'pandas.core.frame.DataFrame'>
Index: 545 entries, 0 to 553
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   school_name  545 non-null    object 
 1   latitude     545 non-null    float64
 2   longitude    545 non-null    float64
dtypes: float64(2), object(1)
memory usage: 17.0+ KB


,school_name,latitude,longitude
0,Maidla Kool,59.336290,27.013928
1,Padise põhikool,59.225410,24.148061
2,Räpina aianduskool,58.096017,27.446897
3,Lüllemäe põhikool,57.753006,26.379099
4,Lauka põhikool,58.962833,22.523060


In [77]:
# Combine both dataframe to get the location of our target schools
df_raw_school_hub_distance = df_raw_school_list.merge(df_raw_schools_coordination, how='left', on='school_name')
df_raw_school_hub_distance.info()
df_raw_school_hub_distance.head()

# Extract missing school cooordination
nan_schools = df_raw_school_hub_distance[df_raw_school_hub_distance["latitude"].isna()]
print(f"Totoal {len(nan_schools)} schools has no info：")
print(nan_schools["school_name"].unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   school_name  155 non-null    object 
 1   latitude     116 non-null    float64
 2   longitude    116 non-null    float64
dtypes: float64(2), object(1)
memory usage: 3.8+ KB
Totoal 39 schools has no info：
['Iisaku Põhikool' 'Elva Gümnaasium' 'Rõngu Keskkool' 'Tabasalu Kool'
 'Oskar Lutsu Palamuse Põhikool' 'Järva-Jaani Kool' 'Põhja-Järva Kool'
 'Kadrina Keskkool' 'Pakri Kool' 'Kullamaa-Risti Keskkool'
 'Lääneranna Gümnaasium' 'Avinurme Gümnaasium' 'Narva Kesklinna Kool'
 'Narva Kreenholmi Kool' 'Narva Pähklimäe Kool' 'Narva Soldino Kool'
 'Vändra Gümnaasium' 'Tõstamaa Keskkool' 'Jüri Kool'
 'Kuressaare Hariduse Kool' 'Setomaa Gümnaasium' 'Tallinna 53. Keskkool'
 'Tallinna Mustamäe Humanitaargümnaasium' 'Tamsalu Kool'
 'Tapa Valla Gümnaasium' 'Väike-Maarja Gümnaasium' 'Hiiumaa Gümnaasium'
 'Kiviõli

In [78]:
# Manually checked missing school coordination
missing_school_coords_manual = {
    "Iisaku Põhikool": (59.092, 27.502),
    "Elva Gümnaasium": (58.2225, 26.4215),
    "Rõngu Keskkool": (58.1397, 26.2517),
    "Tabasalu Kool": (59.386, 24.565),
    "Oskar Lutsu Palamuse Põhikool": (58.6417, 26.5583),
    "Järva-Jaani Kool": (59.0833, 25.8833),
    "Põhja-Järva Kool": (59.185, 25.755),                  # LOW
    "Kadrina Keskkool": (59.3325, 26.1197),
    "Pakri Kool": (59.3567, 24.0536),
    "Kullamaa-Risti Keskkool": (58.994, 23.876),           # LOW
    "Lääneranna Gümnaasium": (58.684, 23.838),
    "Avinurme Gümnaasium": (58.999, 26.962),
    "Narva Kesklinna Kool": (59.377, 28.190),
    "Narva Kreenholmi Kool": (59.379, 28.201),
    "Narva Pähklimäe Kool": (59.372, 28.190),              # LOW
    "Narva Soldino Kool": (59.380, 28.180),                # LOW
    "Vändra Gümnaasium": (58.6483, 25.0378),
    "Tõstamaa Keskkool": (58.2878, 24.1558),
    "Jüri Kool": (59.356, 25.008),
    "Kuressaare Hariduse Kool": (58.2481, 22.5039),
    "Setomaa Gümnaasium": (57.9511, 27.6244),
    "Tallinna 53. Keskkool": (59.437, 24.753),             # LOW
    "Tallinna Mustamäe Humanitaargümnaasium": (59.400, 24.700),  # LOW
    "Tamsalu Kool": (59.1717, 26.0883),
    "Tapa Valla Gümnaasium": (59.2594, 25.9583),
    "Väike-Maarja Gümnaasium": (59.1306, 26.2250),
    "Hiiumaa Gümnaasium": (58.9975, 22.7478),
    "Kiviõli I Keskkool": (59.3553, 26.9728),              # Viru tn 14 (confirmed address)
    "Kiviõli Riigikool": (59.3553, 26.9728),               # same building
    "Narva Eesti Põhikool": (59.378, 28.185),              # LOW
    "Narva Vanalinna Põhikool": (59.377, 28.200),          # LOW
    "Põlva Gümnaasium": (58.0536, 27.0556),
    "Tallinna Mustamäe Riigigümnaasium": (59.3965, 24.6650),  # Akadeemia tee 25 (confirmed address)
    "Tallinna Muusika- ja Balletikool": (59.430, 24.737),  # LOW
    "Tallinna Muusikakeskkool": (59.430, 24.737),          # LOW, merged into Muusika- ja Balletikool
    "Tallinna Tõnismäe Riigigümnaasium": (59.429, 24.737), # LOW
    "Tartu Tähtvere Riigikool": (58.388, 26.700),          # LOW
    "Valga Gümnaasium": (57.7778, 26.0472),
    "Viimsi Gümnaasium": (59.500, 24.830),
}

# Fill the missing latitude and longtitude
manual_lat = df_raw_school_hub_distance["school_name"].map(
    {name: coords[0] for name, coords in missing_school_coords_manual.items()}
)
manual_lon = df_raw_school_hub_distance["school_name"].map(
    {name: coords[1] for name, coords in missing_school_coords_manual.items()}
)

df_raw_school_hub_distance["latitude"] = df_raw_school_hub_distance["latitude"].fillna(manual_lat)
df_raw_school_hub_distance["longitude"] = df_raw_school_hub_distance["longitude"].fillna(manual_lon)
df_raw_school_hub_distance.info()
df_raw_school_hub_distance.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   school_name  155 non-null    object 
 1   latitude     155 non-null    float64
 2   longitude    155 non-null    float64
dtypes: float64(2), object(1)
memory usage: 3.8+ KB


,school_name,latitude,longitude
0,Iisaku Põhikool,59.092000,27.502000
1,Kehra Gümnaasium,59.332499,25.333432
2,Antsla Gümnaasium,57.831820,26.531382
3,Elva Gümnaasium,58.222500,26.421500
4,Rõngu Keskkool,58.139700,26.251700


In [79]:
import geopandas as gpd
from shapely.geometry import Point

# Calculating the distance to major hub (Tallin, Tartu), choose the closest one
df_raw_school_hub_distance.insert(3, column="distance_to_nearest_hub", value=0)


# Hub coordination in WGS84
hubs = {
    "Tallinn": Point(24.7536, 59.4370),
    "Tartu": Point(26.7290, 58.3870),
}

# Distance calculation function using geopandas
gdf_raw_school_hub_distance = gpd.GeoDataFrame(
    df_raw_school_hub_distance,
    geometry=gpd.points_from_xy(df_raw_school_hub_distance["longitude"], df_raw_school_hub_distance["latitude"]),
    crs="EPSG:4326",
).to_crs("EPSG:3301")


gdf_hubs = gpd.GeoDataFrame(
    index=hubs.keys(), geometry=list(hubs.values()), crs="EPSG:4326"
).to_crs("EPSG:3301")


def get_min_hub_distance(school_geom):
  return gdf_hubs.geometry.distance(school_geom).min() / 1000.0


gdf_raw_school_hub_distance["distance_to_nearest_hub"] = gdf_raw_school_hub_distance.geometry.apply(
    get_min_hub_distance
)
df_raw_school_hub_distance["distance_to_nearest_hub"] = gdf_raw_school_hub_distance["distance_to_nearest_hub"]
df_raw_school_hub_distance.info()
df_raw_school_hub_distance.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   school_name              155 non-null    object 
 1   latitude                 155 non-null    float64
 2   longitude                155 non-null    float64
 3   distance_to_nearest_hub  155 non-null    float64
dtypes: float64(3), object(1)
memory usage: 5.0+ KB


,school_name,latitude,longitude,distance_to_nearest_hub
0,Iisaku Põhikool,59.092000,27.502000,90.385562
1,Kehra Gümnaasium,59.332499,25.333432,34.949614
2,Antsla Gümnaasium,57.831820,26.531382,62.922535
3,Elva Gümnaasium,58.222500,26.421500,25.703699
4,Rõngu Keskkool,58.139700,26.251700,39.287459


# X4 : Teaching Language / Exam Langauge

Ideal Format (Columns)

school_name, academic_year, lang_insruction, take_estonian_exam, take_exam_in_ukranian, take_exam_in_russian

In [80]:
# Loading the school language of instruction data
df_raw_lang_instruction = pd.read_excel(ROOT/RAW/'riigieksam_math_2021-2025_by_school_language.xlsx', header=2)
df_raw_lang_instruction.info()
df_raw_lang_instruction.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1430 entries, 0 to 1429
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Õppeaasta                  1430 non-null   object 
 1   Eksam                      1430 non-null   object 
 2   Õppeasutus                 1430 non-null   object 
 3   Õppekeel                   1430 non-null   object 
 4   Max võimalik punktide arv  1430 non-null   int64  
 5   Sooritajate arv            1430 non-null   int64  
 6   Keskmine                   1241 non-null   float64
 7   Keskmine (%)               1241 non-null   float64
 8   St. Hälve                  1241 non-null   float64
 9   Mediaan                    1241 non-null   float64
 10  AUP                        1241 non-null   float64
 11  ÜUP                        1241 non-null   float64
 12  Min                        1241 non-null   float64
 13  Min punktide saajaid       25 non-null     float

,Õppeaasta,Eksam,Õppeasutus,Õppekeel,Max võimalik punktide arv,Sooritajate arv,Keskmine,Keskmine (%),St. Hälve,Mediaan,AUP,ÜUP,Min,Min punktide saajaid,Max,Max punktide saajaid,B2 (%)
0,2024/25,Matemaatika (riigieksam) (kitsas),Abja Gümnaasium,eesti keel,100,13,35.153846,0.351538,19.398652,28.0,24.608619,45.699073,14.0,NaN,77.0,NaN,NaN
1,2024/25,Matemaatika (riigieksam) (kitsas),Antsla Gümnaasium,eesti keel,100,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024/25,Matemaatika (riigieksam) (kitsas),Audentese Spordigümnaasium,eesti keel,100,15,36.400000,0.364000,16.565023,37.0,28.016942,44.783058,6.0,NaN,72.0,NaN,NaN
3,2024/25,Matemaatika (riigieksam) (kitsas),Audentese Spordigümnaasiumi Otepää filiaal,eesti keel,100,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024/25,Matemaatika (riigieksam) (kitsas),August Kitzbergi nimeline Gümnaasium,eesti keel,100,7,34.142857,0.341429,18.836388,25.0,20.188665,48.097049,19.0,NaN,69.0,NaN,NaN


In [81]:
# Extract Math(lai) only
df_raw_lang_instruction = df_raw_lang_instruction[df_raw_lang_instruction['Eksam'].str.contains('lai')]

# Extract only school names and their languages of instruction
df_raw_lang_instruction = df_raw_lang_instruction[['Õppeasutus', 'Õppeaasta', 'Õppekeel']]
df_raw_lang_instruction.info()
df_raw_lang_instruction.head()

<class 'pandas.core.frame.DataFrame'>
Index: 715 entries, 141 to 1429
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Õppeasutus  715 non-null    object
 1   Õppeaasta   715 non-null    object
 2   Õppekeel    715 non-null    object
dtypes: object(3)
memory usage: 22.3+ KB


,Õppeasutus,Õppeaasta,Õppekeel
141,Abja Gümnaasium,2024/25,eesti keel
142,Antsla Gümnaasium,2024/25,eesti keel
143,Audentese Spordigümnaasium,2024/25,eesti keel
144,Audentese Spordigümnaasiumi Otepää filiaal,2024/25,eesti keel
145,August Kitzbergi nimeline Gümnaasium,2024/25,eesti keel


In [82]:
# Translation of estonian column names to english
df_raw_lang_instruction = df_raw_lang_instruction.rename(columns={
    "Õppeaasta": "academic_year",
    "Õppeasutus": "school_name",
    "Õppekeel": "lang_insruction"
})
df_raw_lang_instruction.head()

,school_name,academic_year,lang_insruction
141,Abja Gümnaasium,2024/25,eesti keel
142,Antsla Gümnaasium,2024/25,eesti keel
143,Audentese Spordigümnaasium,2024/25,eesti keel
144,Audentese Spordigümnaasiumi Otepää filiaal,2024/25,eesti keel
145,August Kitzbergi nimeline Gümnaasium,2024/25,eesti keel


In [83]:
# Loading the language of exam data
df_raw_lang_exam = pd.read_excel(ROOT/RAW/'riigieksam_math_2021-2025_by_school_examlanguage.xlsx', header=2)
df_raw_lang_exam.info()
df_raw_lang_exam.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1513 entries, 0 to 1512
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Õppeaasta                  1513 non-null   object 
 1   Eksam                      1513 non-null   object 
 2   Õppeasutus                 1513 non-null   object 
 3   Soorituskeel               1513 non-null   object 
 4   Max võimalik punktide arv  1513 non-null   int64  
 5   Sooritajate arv            1513 non-null   int64  
 6   Keskmine                   1258 non-null   float64
 7   Keskmine (%)               1258 non-null   float64
 8   St. Hälve                  1258 non-null   float64
 9   Mediaan                    1258 non-null   float64
 10  AUP                        1258 non-null   float64
 11  ÜUP                        1258 non-null   float64
 12  Min                        1258 non-null   float64
 13  Min punktide saajaid       24 non-null     float

,Õppeaasta,Eksam,Õppeasutus,Soorituskeel,Max võimalik punktide arv,Sooritajate arv,Keskmine,Keskmine (%),St. Hälve,Mediaan,AUP,ÜUP,Min,Min punktide saajaid,Max,Max punktide saajaid,B2 (%)
0,2024/25,Matemaatika (riigieksam) (kitsas),Abja Gümnaasium,Eesti keel,100,13,35.153846,0.351538,19.398652,28.0,24.608619,45.699073,14.0,NaN,77.0,NaN,NaN
1,2024/25,Matemaatika (riigieksam) (kitsas),Antsla Gümnaasium,Eesti keel,100,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024/25,Matemaatika (riigieksam) (kitsas),Audentese Spordigümnaasium,Eesti keel,100,15,36.400000,0.364000,16.565023,37.0,28.016942,44.783058,6.0,NaN,72.0,NaN,NaN
3,2024/25,Matemaatika (riigieksam) (kitsas),Audentese Spordigümnaasiumi Otepää filiaal,Eesti keel,100,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024/25,Matemaatika (riigieksam) (kitsas),August Kitzbergi nimeline Gümnaasium,Eesti keel,100,7,34.142857,0.341429,18.836388,25.0,20.188665,48.097049,19.0,NaN,69.0,NaN,NaN


In [84]:
# Extract Math(lai) only
df_raw_lang_exam = df_raw_lang_exam [df_raw_lang_exam['Eksam'].str.contains('lai')]

# Extract only school names and their languages of instruction
df_raw_lang_exam = df_raw_lang_exam[['Õppeaasta', 'Õppeasutus', 'Soorituskeel']]

df_raw_lang_exam.info()
df_raw_lang_exam.head()

<class 'pandas.core.frame.DataFrame'>
Index: 758 entries, 155 to 1512
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Õppeaasta     758 non-null    object
 1   Õppeasutus    758 non-null    object
 2   Soorituskeel  758 non-null    object
dtypes: object(3)
memory usage: 23.7+ KB


,Õppeaasta,Õppeasutus,Soorituskeel
155,2024/25,Abja Gümnaasium,Eesti keel
156,2024/25,Antsla Gümnaasium,Eesti keel
157,2024/25,Audentese Spordigümnaasium,Eesti keel
158,2024/25,Audentese Spordigümnaasiumi Otepää filiaal,Eesti keel
159,2024/25,August Kitzbergi nimeline Gümnaasium,Eesti keel


In [85]:
# Translation of estonian column names to english
df_raw_lang_exam = df_raw_lang_exam.rename(columns={
    "Õppeaasta": "academic_year",
    "Õppeasutus": "school_name",
    "Soorituskeel": "exam_language"
})
df_raw_lang_exam.head()

,academic_year,school_name,exam_language
155,2024/25,Abja Gümnaasium,Eesti keel
156,2024/25,Antsla Gümnaasium,Eesti keel
157,2024/25,Audentese Spordigümnaasium,Eesti keel
158,2024/25,Audentese Spordigümnaasiumi Otepää filiaal,Eesti keel
159,2024/25,August Kitzbergi nimeline Gümnaasium,Eesti keel


In [86]:
# Adding extra column as language indicator
df_raw_lang_exam.insert(3, 'flag', value=1)
df_raw_lang_exam.head()

# Pivot the table to mathcing ideal format definted above
df_raw_lang_exam_wide = pd.pivot_table(
    df_raw_lang_exam,
    index=['school_name', 'academic_year'],
    columns='exam_language',
    values='flag'
).reset_index()

df_raw_lang_exam_wide.info()
df_raw_lang_exam_wide.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   school_name    708 non-null    object 
 1   academic_year  708 non-null    object 
 2   Eesti keel     580 non-null    float64
 3   Ukraina keel   2 non-null      float64
 4   Vene keel      176 non-null    float64
dtypes: float64(3), object(2)
memory usage: 27.8+ KB


exam_language,school_name,academic_year,Eesti keel,Ukraina keel,Vene keel
0,''Eduson'' kool,2022/23,NaN,NaN,1.0
1,Abja Gümnaasium,2020/21,1.0,NaN,NaN
2,Abja Gümnaasium,2021/22,1.0,NaN,NaN
3,Abja Gümnaasium,2022/23,1.0,NaN,NaN
4,Abja Gümnaasium,2023/24,1.0,NaN,NaN


In [87]:
# Fill missing flag with 0
df_raw_lang_exam_wide['Eesti keel'] = df_raw_lang_exam_wide['Eesti keel'].fillna(0)
df_raw_lang_exam_wide['Ukraina keel'] = df_raw_lang_exam_wide['Ukraina keel'].fillna(0)
df_raw_lang_exam_wide['Vene keel'] = df_raw_lang_exam_wide['Vene keel'].fillna(0)

df_raw_lang_exam_wide.columns = [
    'school_name',
    'academic_year',
    'take_estonian_exam',
    'take_exam_in_ukranian',
    'take_exam_in_russian'
]
df_raw_lang_exam_wide.info()
df_raw_lang_exam_wide.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   school_name            708 non-null    object 
 1   academic_year          708 non-null    object 
 2   take_estonian_exam     708 non-null    float64
 3   take_exam_in_ukranian  708 non-null    float64
 4   take_exam_in_russian   708 non-null    float64
dtypes: float64(3), object(2)
memory usage: 27.8+ KB


,school_name,academic_year,take_estonian_exam,take_exam_in_ukranian,take_exam_in_russian
0,''Eduson'' kool,2022/23,0.0,0.0,1.0
1,Abja Gümnaasium,2020/21,1.0,0.0,0.0
2,Abja Gümnaasium,2021/22,1.0,0.0,0.0
3,Abja Gümnaasium,2022/23,1.0,0.0,0.0
4,Abja Gümnaasium,2023/24,1.0,0.0,0.0


In [88]:
# Combine two table regarding used languages
df_used_language_combined = df_raw_lang_instruction.merge(df_raw_lang_exam_wide, how='inner', on=['school_name', 'academic_year'])
df_used_language_combined.head()

,school_name,academic_year,lang_insruction,take_estonian_exam,take_exam_in_ukranian,take_exam_in_russian
0,Abja Gümnaasium,2024/25,eesti keel,1.0,0.0,0.0
1,Antsla Gümnaasium,2024/25,eesti keel,1.0,0.0,0.0
2,Audentese Spordigümnaasium,2024/25,eesti keel,1.0,0.0,0.0
3,Audentese Spordigümnaasiumi Otepää filiaal,2024/25,eesti keel,1.0,0.0,0.0
4,August Kitzbergi nimeline Gümnaasium,2024/25,eesti keel,1.0,0.0,0.0


In [89]:
df_used_language_combined[
    (df_used_language_combined['lang_insruction'] == 'eesti keel') &
    (df_used_language_combined['take_exam_in_russian'] != 0.0)
]

,school_name,academic_year,lang_insruction,take_estonian_exam,take_exam_in_ukranian,take_exam_in_russian
5,Avinurme Gümnaasium,2024/25,eesti keel,1.0,0.0,1.0
9,Erakool Garant,2024/25,eesti keel,0.0,0.0,1.0
19,Jõhvi Gümnaasium,2024/25,eesti keel,1.0,0.0,1.0
28,Kiviõli Riigikool,2024/25,eesti keel,1.0,0.0,1.0
40,Lasnamäe Gümnaasium,2024/25,eesti keel,0.0,0.0,1.0
...,...,...,...,...,...,...
681,Tallinna Õismäe Vene Lütseum,2020/21,eesti keel,0.0,0.0,1.0
682,Tallinna Pae Gümnaasium,2020/21,eesti keel,0.0,0.0,1.0
687,Tallinna Tõnismäe Reaalkool,2020/21,eesti keel,0.0,0.0,1.0
696,Tartu Kristjan Jaak Petersoni Gümnaasium,2020/21,eesti keel,1.0,0.0,1.0


# X5 : School area income level (average ,median)

Ideal Format (Columns)

school_name,	county,	municipality,	2021,	2022,	2023,	2024,	2025



In [90]:
# Loading the school location data
df_raw_school_location = pd.read_excel(ROOT/RAW/'riigieksam_math_2021-2025_by_school_location.xlsx', header=2)
df_raw_school_location.info()
df_raw_school_location.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1419 entries, 0 to 1418
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Õppeaasta                  1419 non-null   object 
 1   Eksam                      1419 non-null   object 
 2   Õppeasutus                 1419 non-null   object 
 3   Maakond                    1419 non-null   object 
 4   Kohalik omavalitsus        1419 non-null   object 
 5   Õppeasutuse tüüp           1419 non-null   object 
 6   Max võimalik punktide arv  1419 non-null   int64  
 7   Sooritajate arv            1419 non-null   int64  
 8   Keskmine                   1240 non-null   float64
 9   Keskmine (%)               1240 non-null   float64
 10  St. Hälve                  1240 non-null   float64
 11  Mediaan                    1240 non-null   float64
 12  AUP                        1240 non-null   float64
 13  ÜUP                        1240 non-null   float

,Õppeaasta,Eksam,Õppeasutus,Maakond,Kohalik omavalitsus,Õppeasutuse tüüp,Max võimalik punktide arv,Sooritajate arv,Keskmine,Keskmine (%),St. Hälve,Mediaan,AUP,ÜUP,Min,Min punktide saajaid,Max,Max punktide saajaid,B2 (%)
0,2024/25,Matemaatika (riigieksam) (kitsas),Abja Gümnaasium,Viljandi maakond,Mulgi vald,põhikool või gümnaasium,100,13,35.153846,0.351538,19.398652,28.0,24.608619,45.699073,14.0,NaN,77.0,NaN,NaN
1,2024/25,Matemaatika (riigieksam) (kitsas),Antsla Gümnaasium,Võru maakond,Antsla vald,põhikool või gümnaasium,100,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024/25,Matemaatika (riigieksam) (kitsas),Audentese Spordigümnaasium,Harju maakond,Tallinn,põhikool või gümnaasium,100,15,36.400000,0.364000,16.565023,37.0,28.016942,44.783058,6.0,NaN,72.0,NaN,NaN
3,2024/25,Matemaatika (riigieksam) (kitsas),Audentese Spordigümnaasiumi Otepää filiaal,Valga maakond,Otepää vald,põhikool või gümnaasium,100,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024/25,Matemaatika (riigieksam) (kitsas),August Kitzbergi nimeline Gümnaasium,Viljandi maakond,Mulgi vald,põhikool või gümnaasium,100,7,34.142857,0.341429,18.836388,25.0,20.188665,48.097049,19.0,NaN,69.0,NaN,NaN


In [91]:
# Extract Math(lai) only
df_raw_school_location = df_raw_school_location[df_raw_school_location['Eksam'].str.contains('lai')]

# Extract only school names and corresponded counties and municipalities
df_raw_school_location= df_raw_school_location[['Õppeasutus', 'Maakond', 'Kohalik omavalitsus']]
df_raw_school_location.info()
df_raw_school_location.head()

<class 'pandas.core.frame.DataFrame'>
Index: 708 entries, 141 to 1418
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Õppeasutus           708 non-null    object
 1   Maakond              708 non-null    object
 2   Kohalik omavalitsus  708 non-null    object
dtypes: object(3)
memory usage: 22.1+ KB


,Õppeasutus,Maakond,Kohalik omavalitsus
141,Abja Gümnaasium,Viljandi maakond,Mulgi vald
142,Antsla Gümnaasium,Võru maakond,Antsla vald
143,Audentese Spordigümnaasium,Harju maakond,Tallinn
144,Audentese Spordigümnaasiumi Otepää filiaal,Valga maakond,Otepää vald
145,August Kitzbergi nimeline Gümnaasium,Viljandi maakond,Mulgi vald


In [92]:
# Translation of estonian column names to english
df_raw_school_location = df_raw_school_location.rename(columns={
    "Õppeasutus": "school_name",
    "Maakond": "county",
    "Kohalik omavalitsus": "municipality"
})
df_raw_school_location.head()

,school_name,county,municipality
141,Abja Gümnaasium,Viljandi maakond,Mulgi vald
142,Antsla Gümnaasium,Võru maakond,Antsla vald
143,Audentese Spordigümnaasium,Harju maakond,Tallinn
144,Audentese Spordigümnaasiumi Otepää filiaal,Valga maakond,Otepää vald
145,August Kitzbergi nimeline Gümnaasium,Viljandi maakond,Mulgi vald


In [93]:
# Loading the municipality average income data
df_raw_avg_income = pd.read_excel(ROOT/RAW/'stat_income_avg_municipality.xlsx')
df_raw_avg_income.columns = ['register Number', 'municipality', '2021', '2022', '2023', '2024', '2025']
df_raw_avg_income = df_raw_avg_income.drop(columns=['register Number'])
df_raw_avg_income.info()
df_raw_avg_income.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   municipality  101 non-null    object
 1   2021          101 non-null    object
 2   2022          101 non-null    object
 3   2023          101 non-null    object
 4   2024          101 non-null    object
 5   2025          101 non-null    object
dtypes: object(6)
memory usage: 4.9+ KB


,municipality,2021,2022,2023,2024,2025
0,Eesti,1474,1645,1832,1981,2092
1,Harju maakond,1658,1855,2057,2218,2336
2,Anija vald,1246,1397,1446,1584,1663
3,Harku vald,1381,1515,1635,1733,1845
4,Jõelähtme vald,1409,1520,1667,1753,1879


In [94]:
# Mapping the average income by municipality of school
df_raw_school_avg_income = df_raw_school_location.merge(df_raw_avg_income, on="municipality", how='left')
df_raw_school_avg_income.info()
df_raw_school_avg_income.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   school_name   708 non-null    object
 1   county        708 non-null    object
 2   municipality  708 non-null    object
 3   2021          708 non-null    object
 4   2022          708 non-null    object
 5   2023          708 non-null    object
 6   2024          708 non-null    object
 7   2025          708 non-null    object
dtypes: object(8)
memory usage: 44.4+ KB


,school_name,county,municipality,2021,2022,2023,2024,2025
0,Abja Gümnaasium,Viljandi maakond,Mulgi vald,1164,1304,1434,1532,1616
1,Antsla Gümnaasium,Võru maakond,Antsla vald,..,..,..,1355,1427
2,Audentese Spordigümnaasium,Harju maakond,Tallinn,1724,1935,2152,2325,2451
3,Audentese Spordigümnaasiumi Otepää filiaal,Valga maakond,Otepää vald,1125,1211,1390,1504,1604
4,August Kitzbergi nimeline Gümnaasium,Viljandi maakond,Mulgi vald,1164,1304,1434,1532,1616


In [95]:
# Loading the municipality median income data
df_raw_median_income = pd.read_excel(ROOT/RAW/'stat_income_median_municipalily.xlsx')
df_raw_median_income.columns = ['register Number', 'municipality', '2021', '2022', '2023', '2024', '2025']
df_raw_median_income = df_raw_median_income.drop(columns=['register Number'])
df_raw_median_income.info()
df_raw_median_income.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   municipality  100 non-null    object
 1   2021          100 non-null    object
 2   2022          100 non-null    object
 3   2023          100 non-null    object
 4   2024          100 non-null    object
 5   2025          100 non-null    object
dtypes: object(6)
memory usage: 4.8+ KB


,municipality,2021,2022,2023,2024,2025
0,Harju maakond,1340,1500,1685,1818,1916
1,Anija vald,1107,1230,1281,1386,1467
2,Harku vald,1215,1352,1450,1544,1638
3,Jõelähtme vald,1207,1308,1417,1489,1595
4,Keila linn,1166,1262,1352,1490,1553


In [96]:
# Mapping the median income by municipality of school
df_raw_school_median_income = df_raw_school_location.merge(df_raw_median_income, on="municipality", how='left')
df_raw_school_median_income.info()
df_raw_school_median_income.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   school_name   708 non-null    object
 1   county        708 non-null    object
 2   municipality  708 non-null    object
 3   2021          708 non-null    object
 4   2022          708 non-null    object
 5   2023          708 non-null    object
 6   2024          708 non-null    object
 7   2025          708 non-null    object
dtypes: object(8)
memory usage: 44.4+ KB


,school_name,county,municipality,2021,2022,2023,2024,2025
0,Abja Gümnaasium,Viljandi maakond,Mulgi vald,996,1118,1224,1308,1387
1,Antsla Gümnaasium,Võru maakond,Antsla vald,..,..,..,1155,1205
2,Audentese Spordigümnaasium,Harju maakond,Tallinn,1378,1543,1749,1896,2000
3,Audentese Spordigümnaasiumi Otepää filiaal,Valga maakond,Otepää vald,1000,1061,1238,1333,1420
4,August Kitzbergi nimeline Gümnaasium,Viljandi maakond,Mulgi vald,996,1118,1224,1308,1387
